In [85]:
import nflreadpy as nfl
import pandas as pd

# Load weekly player stats for 2019-2025
player_stats = nfl.load_player_stats(range(2019,2026))

# Convert from Polars to pandas
df = player_stats.to_pandas()

print(df.shape)
print(df['position'].value_counts())
print(df['season'].unique())

(129812, 150)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'game_id', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving

In [86]:
# What positions exist and how many rows per position
print(df['position'].value_counts())

# Confirm season/week ranges
print(df['season'].unique())
print(df['week'].unique())

# Check for missing values in key fantasy-relevant columns
key_cols = ['player_display_name', 'position', 'team', 'opponent_team',
            'carries', 'targets', 'receptions', 'fantasy_points', 'fantasy_points_ppr']
print(df[key_cols].isnull().sum())

position
WR     17683
LB     15492
CB     13329
RB     11087
DE     10828
DT     10108
TE      8740
SAF     6311
QB      4690
DB      4478
K       3914
P       3865
OT      3147
OLB     3062
FS      2487
G       2129
S       1829
ILB     1531
MLB     1437
C       1077
NT       930
FB       748
LS       482
DL       232
OL        44
Name: count, dtype: int64
[2019 2020 2021 2022 2023 2024 2025]
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22]
player_display_name    152
position               152
team                     0
opponent_team            0
carries                  0
targets                  0
receptions               0
fantasy_points           0
fantasy_points_ppr       0
dtype: int64


In [87]:
# Sorting dataset into players and seasons (chronologically)
df = df.sort_values(['player_id','season','week']).reset_index(drop=True)


In [88]:
# ----------- WR/TE Feature Engineering -----------

# Past 3 game averages
df['targets_avg_3'] = df.groupby(['player_id', 'season'])['targets'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_avg_3'] = df.groupby(['player_id', 'season'])['receptions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_yards_avg_3'] = df.groupby(['player_id', 'season'])['receiving_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['target_share_avg_3'] = df.groupby(['player_id','season'])['target_share'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_air_yards_avg_3'] = df.groupby(['player_id','season'])['receiving_air_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['air_yards_share_avg_3'] = df.groupby(['player_id','season'])['air_yards_share'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_td_avg_3'] = df.groupby(['player_id','season'])['receiving_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_yards_after_catch_avg_3'] = df.groupby(['player_id','season'])['receiving_yards_after_catch'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['targets_avg_5'] = df.groupby(['player_id', 'season'])['targets'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_avg_5'] = df.groupby(['player_id', 'season'])['receptions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_yards_avg_5'] = df.groupby(['player_id', 'season'])['receiving_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['target_share_avg_5'] = df.groupby(['player_id','season'])['target_share'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_air_yards_avg_5'] = df.groupby(['player_id','season'])['receiving_air_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['air_yards_share_avg_5'] = df.groupby(['player_id','season'])['air_yards_share'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_td_avg_5'] = df.groupby(['player_id','season'])['receiving_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_yards_after_catch_avg_5'] = df.groupby(['player_id','season'])['receiving_yards_after_catch'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['targets_trend'] = df['targets_avg_3'] - df['targets_avg_5']
df['target_share_trend'] = df['target_share_avg_3'] - df['target_share_avg_5']
df['rec_yards_trend'] = df['rec_yards_avg_3'] - df['rec_yards_avg_5']
df['rec_air_yards_trend'] = df['rec_air_yards_avg_3'] - df['rec_air_yards_avg_5']



In [89]:
# ----------- RB Feature Engineering -----------

df['opportunities'] = df['carries'] + df['targets']

# Past 3 game averages
df['carries_avg_3'] = df.groupby(['player_id','season'])['carries'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rushing_yards_avg_3'] = df.groupby(['player_id','season'])['rushing_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rushing_tds_avg_3'] = df.groupby(['player_id','season'])['rushing_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['opportunities_avg_3'] = df.groupby(['player_id','season'])['opportunities'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())


# Past 5 game averages
df['carries_avg_5'] = df.groupby(['player_id','season'])['carries'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rushing_yards_avg_5'] = df.groupby(['player_id','season'])['rushing_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rushing_tds_avg_5'] = df.groupby(['player_id','season'])['rushing_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['opportunities_avg_5'] = df.groupby(['player_id','season'])['opportunities'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['carries_trend'] = df['carries_avg_3'] - df['carries_avg_5']
df['rushing_yards_trend'] = df['rushing_yards_avg_3'] - df['rushing_yards_avg_5']
df['opportunities_trend'] = df['opportunities_avg_3'] - df['opportunities_avg_5']


In [90]:
# ----------- QB Feature Engineering -----------

# Past 3 game averages
df['completions_avg_3'] = df.groupby(['player_id', 'season'])['completions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['attempts_avg_3'] = df.groupby(['player_id', 'season'])['attempts'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_yards_avg_3'] = df.groupby(['player_id', 'season'])['passing_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_tds_avg_3'] = df.groupby(['player_id', 'season'])['passing_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_int_avg_3'] = df.groupby(['player_id', 'season'])['passing_interceptions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_air_yards_avg_3'] = df.groupby(['player_id', 'season'])['passing_air_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_first_downs_avg_3'] = df.groupby(['player_id', 'season'])['passing_first_downs'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['completions_avg_5'] = df.groupby(['player_id', 'season'])['completions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['attempts_avg_5'] = df.groupby(['player_id', 'season'])['attempts'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_yards_avg_5'] = df.groupby(['player_id', 'season'])['passing_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_tds_avg_5'] = df.groupby(['player_id', 'season'])['passing_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_int_avg_5'] = df.groupby(['player_id', 'season'])['passing_interceptions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_air_yards_avg_5'] = df.groupby(['player_id', 'season'])['passing_air_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_first_downs_avg_5'] = df.groupby(['player_id', 'season'])['passing_first_downs'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['attempts_trend'] = df['attempts_avg_3'] - df['attempts_avg_5']
df['passing_yards_trend'] = df['passing_yards_avg_3'] - df['passing_yards_avg_5']
df['passing_air_yards_trend'] = df['passing_air_yards_avg_3'] - df['passing_air_yards_avg_5']


In [91]:
# ----------- K Feature Engineering -----------

# Past 3 game averages
df['fg_att_avg_3'] = df.groupby(['player_id', 'season'])['fg_att'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_made_avg_3'] = df.groupby(['player_id', 'season'])['fg_made'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_long_avg_3'] = df.groupby(['player_id', 'season'])['fg_long'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_made_50_59_avg_3'] = df.groupby(['player_id', 'season'])['fg_made_50_59'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['pat_att_avg_3'] = df.groupby(['player_id', 'season'])['pat_att'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['pat_made_avg_3'] = df.groupby(['player_id', 'season'])['pat_made'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['fg_att_avg_5'] = df.groupby(['player_id', 'season'])['fg_att'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_made_avg_5'] = df.groupby(['player_id', 'season'])['fg_made'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_long_avg_5'] = df.groupby(['player_id', 'season'])['fg_long'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_made_50_59_avg_5'] = df.groupby(['player_id', 'season'])['fg_made_50_59'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['pat_att_avg_5'] = df.groupby(['player_id', 'season'])['pat_att'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['pat_made_avg_5'] = df.groupby(['player_id', 'season'])['pat_made'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['fg_att_trend'] = df['fg_att_avg_3'] - df['fg_att_avg_5']
df['fg_made_trend'] = df['fg_made_avg_3'] - df['fg_made_avg_5']

In [95]:
# ----------- WR/TE Training Dataset -----------

wr_te_df = df[df['position'].isin(['WR', 'TE'])].copy()
wr_te_features = [
    # Receiving
    'targets_avg_3',
    'targets_avg_5',
    'rec_avg_3',
    'rec_avg_5',
    'rec_yards_avg_3',
    'rec_yards_avg_5',
    'target_share_avg_3',
    'target_share_avg_5',
    'rec_air_yards_avg_3',
    'rec_air_yards_avg_5',
    'air_yards_share_avg_3',
    'air_yards_share_avg_5',
    'rec_td_avg_3',
    'rec_td_avg_5',
    'rec_yards_after_catch_avg_3',
    'rec_yards_after_catch_avg_5',

    # Trends
    'targets_trend',
    'target_share_trend',
    'rec_yards_trend',
    'rec_air_yards_trend'
]

X_wr_te = wr_te_df[wr_te_features]
y_wr_te = wr_te_df['fantasy_points_ppr']

# Removing empty row that has no prior game history
valid_wr_te = X_wr_te.notna().all(axis=1)

X_wr_te = X_wr_te[valid_wr_te]
y_wr_te = y_wr_te[valid_wr_te]


(23859, 20)
(23859,)
0


In [96]:
# ----------- RB Training Dataset -----------

rb_df = df[df['position'] == 'RB'].copy()
rb_features = [
    # Rushing
    'carries_avg_3',
    'carries_avg_5',
    'rushing_yards_avg_3',
    'rushing_yards_avg_5',
    'rushing_tds_avg_3',
    'rushing_tds_avg_5',
    'opportunities_avg_3',
    'opportunities_avg_5',

    # Receiving
    'targets_avg_3',
    'targets_avg_5',
    'rec_avg_3',
    'rec_avg_5',
    'rec_yards_avg_3',
    'rec_yards_avg_5',
    'target_share_avg_3',
    'target_share_avg_5',
    'rec_td_avg_3',
    'rec_td_avg_5',

    # Trends
    'carries_trend',
    'rushing_yards_trend',
    'opportunities_trend',
    'targets_trend',
    'target_share_trend',
    'rec_yards_trend'
]

X_rb = rb_df[rb_features]
y_rb = rb_df['fantasy_points_ppr']

# Removing empty row that has no prior game history
valid_rb = X_rb.notna().all(axis=1)

X_rb = X_rb[valid_rb]
y_rb = y_rb[valid_rb]


In [136]:
# ----------- QB Training Dataset -----------

qb_df = df[df['position'] == 'QB'].copy()

qb_features = [
    # Passing
    'completions_avg_3',
    'completions_avg_5',
    'attempts_avg_3',
    'attempts_avg_5',
    'passing_yards_avg_3',
    'passing_yards_avg_5',
    'passing_tds_avg_3',
    'passing_tds_avg_5',
    'passing_int_avg_3',
    'passing_int_avg_5',
    'passing_air_yards_avg_3',
    'passing_air_yards_avg_5',
    'passing_first_downs_avg_3',
    'passing_first_downs_avg_5',

    # Rushing
    'carries_avg_3',
    'carries_avg_5',
    'rushing_yards_avg_3',
    'rushing_yards_avg_5',
    'rushing_tds_avg_3',
    'rushing_tds_avg_5',

    # Trends
    'attempts_trend',
    'passing_yards_trend',
    'passing_air_yards_trend',
    'carries_trend',
    'rushing_yards_trend'
]

X_qb = qb_df[qb_features]
y_qb = qb_df['fantasy_points_ppr']

valid_qb = X_qb.notna().all(axis=1)

X_qb = X_qb[valid_qb]
y_qb = y_qb[valid_qb]

In [137]:
# ----------- K Training Dataset -----------

k_df = df[df['position'] == 'K'].copy()

k_features = [
    'fg_att_avg_3',
    'fg_att_avg_5',
    'fg_made_avg_3',
    'fg_made_avg_5',
    'fg_long_avg_3',
    'fg_long_avg_5',
    'fg_made_50_59_avg_3',
    'fg_made_50_59_avg_5',
    'pat_att_avg_3',
    'pat_att_avg_5',
    'pat_made_avg_3',
    'pat_made_avg_5',
    'fg_att_trend',
    'fg_made_trend'
]

k_df['kicker_fantasy_points'] = (
    3 * k_df['fg_made'] +
    1 * k_df['pat_made']
)

X_k = k_df[k_features]
y_k = k_df['kicker_fantasy_points']

valid_k = X_k.notna().all(axis=1)

X_k = X_k[valid_k]
y_k = y_k[valid_k]


In [102]:
# ----------- WR/TE Training, Validation, Test Split -----------

wr_te_seasons = wr_te_df.loc[X_wr_te.index, 'season']

train_mask_wr_te = wr_te_seasons <= 2023 # train / learn from past seasons
val_mask_wr_te = wr_te_seasons == 2024 # validate / tune from later season
test_mask_wr_te = wr_te_seasons == 2025 # test / final eval from from latest season

X_train_wr_te = X_wr_te[train_mask_wr_te]
y_train_wr_te = y_wr_te[train_mask_wr_te]

X_val_wr_te = X_wr_te[val_mask_wr_te]
y_val_wr_te = y_wr_te[val_mask_wr_te]

X_test_wr_te = X_wr_te[test_mask_wr_te]
y_test_wr_te = y_wr_te[test_mask_wr_te]

print(X_train_wr_te.shape)
print(X_val_wr_te.shape)
print(X_test_wr_te.shape)

(16789, 20)
(3469, 20)
(3601, 20)


In [104]:
# ----------- WR/TE Linear Regression Model -----------

from sklearn.linear_model import LinearRegression

wr_te_linear_model = LinearRegression()

wr_te_linear_model.fit(
    X_train_wr_te,
    y_train_wr_te
)

# Take model learned from 2019-2023 to get PREDICTED fantasy points for every WR/TE row in 2024
y_val_pred_wr_te = wr_te_linear_model.predict(X_val_wr_te)

from sklearn.metrics import mean_absolute_error, root_mean_squared_error

mae_wr_te = mean_absolute_error(y_val_wr_te, y_val_pred_wr_te)
rmse_wr_te = root_mean_squared_error(y_val_wr_te, y_val_pred_wr_te)

print("WR/TE Validation MAE:", mae_wr_te) # How wrong on average?
print("WR/TE Validation RMSE:", rmse_wr_te) # How much do big mistakes hurt?


In [109]:
# ----------- WR/TE Linear Regression Model -----------

from sklearn.metrics import mean_absolute_error, root_mean_squared_error

mae_wr_te = mean_absolute_error(y_val_wr_te, y_val_pred_wr_te)
rmse_wr_te = root_mean_squared_error(y_val_wr_te, y_val_pred_wr_te)

print("WR/TE Validation MAE:", mae_wr_te) # How wrong on average?
print("WR/TE Validation RMSE:", rmse_wr_te) # How much do big mistakes hurt?


WR/TE Validation MAE: 4.3550197290650825
WR/TE Validation RMSE: 5.98517531182884


In [117]:
# ----------- WR/TE Random Forest Model -----------

from sklearn.ensemble import RandomForestRegressor

wr_te_rf_model = RandomForestRegressor(
    n_estimators=100, # Build 100 decision trees
    random_state=42 # Makes randomness reproducible 
)

wr_te_rf_model.fit(
    X_train_wr_te,
    y_train_wr_te
)

# Predict 2024
y_val_pred_rf_wr_te = wr_te_rf_model.predict(X_val_wr_te)

mae_rf_wr_te = mean_absolute_error(
    y_val_wr_te,
    y_val_pred_rf_wr_te
)

rmse_rf_wr_te = root_mean_squared_error(
    y_val_wr_te,
    y_val_pred_rf_wr_te
)

print("Random Forest WR/TE Validation MAE:", mae_rf_wr_te)
print("Random Forest WR/TE Validation RMSE:", rmse_rf_wr_te)

y_train_pred_rf_wr_te = wr_te_rf_model.predict(X_train_wr_te)

train_mae_rf_wr_te = mean_absolute_error(
    y_train_wr_te,
    y_train_pred_rf_wr_te
)

print("Random Forest Training MAE:", train_mae_rf_wr_te)
print("Random Forest Validation MAE", mae_rf_wr_te)


Random Forest WR/TE Validation MAE: 4.569101092794647
Random Forest WR/TE Validation RMSE: 6.1381267566983535
Random Forest Training MAE: 1.7658297436716968
Random Forest Validation MAE 4.569101092794647
Tuned Random Forest MAE: 4.377710326417749
Tuned Random Forest RMSE: 5.9978006895198055


In [115]:
# ----------- WR/TE XGBoost Model -----------
from xgboost import XGBRegressor

wr_te_xgb_model = XGBRegressor(
    n_estimators = 100,
    random_state=42
)

wr_te_xgb_model.fit(
    X_train_wr_te,
    y_train_wr_te
)

# Predict 2024
y_val_pred_xgb_wr_te = wr_te_xgb_model.predict(X_val_wr_te)

mae_xgb_wr_te = mean_absolute_error(
    y_val_wr_te,
    y_val_pred_xgb_wr_te
)

rmse_xgb_wr_te = root_mean_squared_error(
    y_val_wr_te,
    y_val_pred_xgb_wr_te
)

print("XGBoost WR/TE Validation MAE:", mae_xgb_wr_te)
print("XGBoost WR/TE Validation RMSE:", rmse_xgb_wr_te)


XGBoost WR/TE Validation MAE: 4.596481784364022
XGBoost WR/TE Validation RMSE: 6.327636564601338
Tuned XGBoost MAE: 4.357544846423389
Tuned XGBoost RMSE: 5.977960384695904


In [118]:
# ----------- Tuning Models -----------

# XG Boost

wr_te_xgb_tuned_1 = XGBRegressor(
    n_estimators=300, # More trees
    learning_rate=0.05, # Smaller learning steps
    max_depth=3, # Shallower trees
    subsample=0.8, # Row sampling
    colsample_bytree=0.8, # feature sampling
    random_state=42
)

wr_te_xgb_tuned_1.fit(
    X_train_wr_te,
    y_train_wr_te
)

y_val_pred_xgb_tuned_1 = wr_te_xgb_tuned_1.predict(X_val_wr_te)

mae_xgb_tuned_1 = mean_absolute_error(
    y_val_wr_te,
    y_val_pred_xgb_tuned_1
)

rmse_xgb_tuned_1 = root_mean_squared_error(
    y_val_wr_te,
    y_val_pred_xgb_tuned_1
)

print("Tuned XGBoost MAE:", mae_xgb_tuned_1)
print("Tuned XGBoost RMSE:", rmse_xgb_tuned_1)

# Random Forest

wr_te_rf_tuned_1 = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

wr_te_rf_tuned_1.fit(
    X_train_wr_te,
    y_train_wr_te
)

y_val_pred_rf_tuned_1 = wr_te_rf_tuned_1.predict(X_val_wr_te)

mae_rf_tuned_1 = mean_absolute_error(
    y_val_wr_te,
    y_val_pred_rf_tuned_1
)

rmse_rf_tuned_1 = root_mean_squared_error(
    y_val_wr_te,
    y_val_pred_rf_tuned_1
)

print("Tuned Random Forest MAE:", mae_rf_tuned_1)
print("Tuned Random Forest RMSE:", rmse_rf_tuned_1)

Tuned XGBoost MAE: 4.357544846423389
Tuned XGBoost RMSE: 5.977960384695904
Tuned Random Forest MAE: 4.377710326417749
Tuned Random Forest RMSE: 5.9978006895198055


In [119]:
# ----------- Chronological Setup -----------
wr_te_train_info = wr_te_df.loc[
    X_train_wr_te.index,
    ['season', 'week']
].copy()

chronological_order = wr_te_train_info.sort_values(
    ['season', 'week']
).index

X_train_wr_te_chrono = X_train_wr_te.loc[chronological_order]
y_train_wr_te_chrono = y_train_wr_te.loc[chronological_order]


In [120]:
# ----------- WR/TE Expanding Season Validation Folds -----------

season_series = wr_te_df.loc[
    X_train_wr_te_chrono.index,
    'season'
]

cv_splits = []

for val_season in [2021, 2022, 2023]:
    train_indices = season_series[season_series < val_season].index
    val_indices = season_series[season_series == val_season].index

    train_positions = X_train_wr_te_chrono.index.get_indexer(train_indices)
    val_positions = X_train_wr_te_chrono.index.get_indexer(val_indices)

    cv_splits.append((train_positions, val_positions))

for i, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    print(
        f"Fold {i}:",
        len(train_idx),
        "train rows,",
        len(val_idx),
        "validation rows"
    )

Fold 1: 6423 train rows, 3471 validation rows
Fold 2: 9894 train rows, 3402 validation rows
Fold 3: 13296 train rows, 3493 validation rows


In [132]:
# ----------- WR/TE XGBoost Grid Search -----------

from sklearn.model_selection import GridSearchCV

wr_param_grid = {
    'n_estimators': [200, 300],
    'learning_rate': [0.03, 0.05],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=wr_param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv_splits,
    n_jobs=-1 # flipped because higher score = worse
)

xgb_grid.fit(
    X_train_wr_te_chrono,
    y_train_wr_te_chrono
)

print("Best parameters:", xgb_grid.best_params_)
print("Best CV MAE:", -xgb_grid.best_score_)

best_xgb_model = xgb_grid.best_estimator_

y_val_pred_best_xgb = best_xgb_model.predict(X_val_wr_te)

best_xgb_val_mae = mean_absolute_error(
    y_val_wr_te,
    y_val_pred_best_xgb
)

best_xgb_val_rmse = root_mean_squared_error(
    y_val_wr_te,
    y_val_pred_best_xgb
)

print("Best XGBoost 2024 Validation MAE:", best_xgb_val_mae)
print("Best XGBoost 2024 Validation RMSE:", best_xgb_val_rmse)

Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 300, 'subsample': 0.8}
Best CV MAE: 4.447321438979841
Best XGBoost 2024 Validation MAE: 4.341026478440523
Best XGBoost 2024 Validation RMSE: 5.9669827482029545


In [123]:
# ----------- RB Training, Validation, Test Split -----------

rb_seasons = rb_df.loc[X_rb.index, 'season']

train_mask_rb = rb_seasons <= 2023
val_mask_rb = rb_seasons == 2024
test_mask_rb = rb_seasons == 2025

X_train_rb = X_rb[train_mask_rb]
y_train_rb = y_rb[train_mask_rb]

X_val_rb = X_rb[val_mask_rb]
y_val_rb = y_rb[val_mask_rb]

X_test_rb = X_rb[test_mask_rb]
y_test_rb = y_rb[test_mask_rb]

print(X_train_rb.shape)
print(X_val_rb.shape)
print(X_test_rb.shape)

(7053, 24)
(1462, 24)
(1496, 24)


In [124]:
# ----------- RB Linear Regression Model -----------

rb_linear_model = LinearRegression()

rb_linear_model.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_rb = rb_linear_model.predict(X_val_rb)

mae_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_rb
)

rmse_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_rb
)

print("RB Linear Regression Validation MAE:", mae_rb)
print("RB Linear Regression Validation RMSE:", rmse_rb)

RB Linear Regression Validation MAE: 4.554592037889293
RB Linear Regression Validation RMSE: 6.128899056336454


In [125]:
# ----------- RB Random Forest Model -----------

rb_rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rb_rf_model.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_rf_rb = rb_rf_model.predict(X_val_rb)

mae_rf_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_rf_rb
)

rmse_rf_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_rf_rb
)

print("RB Random Forest Validation MAE:", mae_rf_rb)
print("RB Random Forest Validation RMSE:", rmse_rf_rb)

RB Random Forest Validation MAE: 4.773985458557423
RB Random Forest Validation RMSE: 6.362547766519092


In [126]:
# ----------- RB XGBoost Model -----------

rb_xgb_model = XGBRegressor(
    n_estimators=100,
    random_state=42
)

rb_xgb_model.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_xgb_rb = rb_xgb_model.predict(X_val_rb)

mae_xgb_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_xgb_rb
)

rmse_xgb_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_xgb_rb
)

print("RB XGBoost Validation MAE:", mae_xgb_rb)
print("RB XGBoost Validation RMSE:", rmse_xgb_rb)

RB XGBoost Validation MAE: 4.894608853202104
RB XGBoost Validation RMSE: 6.67406707247599


In [129]:
# ----------- Tuning Models -----------

# Random Forest
rb_rf_tuned_1 = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

rb_rf_tuned_1.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_rf_tuned_rb = rb_rf_tuned_1.predict(X_val_rb)

mae_rf_tuned_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_rf_tuned_rb
)

rmse_rf_tuned_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_rf_tuned_rb
)

print("Tuned RB Random Forest MAE:", mae_rf_tuned_rb)
print("Tuned RB Random Forest RMSE:", rmse_rf_tuned_rb)

# XGBoost
rb_xgb_tuned_1 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

rb_xgb_tuned_1.fit(
    X_train_rb,
    y_train_rb
)

y_val_pred_xgb_tuned_rb = rb_xgb_tuned_1.predict(X_val_rb)

mae_xgb_tuned_rb = mean_absolute_error(
    y_val_rb,
    y_val_pred_xgb_tuned_rb
)

rmse_xgb_tuned_rb = root_mean_squared_error(
    y_val_rb,
    y_val_pred_xgb_tuned_rb
)

print("tuned RB XGBoost MAE:", mae_xgb_tuned_rb)
print("Tuned RB XGBoost RMSE:", rmse_xgb_tuned_rb)

Tuned RB Random Forest MAE: 4.60490797828784
Tuned RB Random Forest RMSE: 6.172493197762412
tuned RB XGBoost MAE: 4.560298755684237
Tuned RB XGBoost RMSE: 6.144054412013649


In [130]:
# ----------- RB Chronological Ordering for Model Selection -----------

rb_train_info = rb_df.loc[
    X_train_rb.index,
    ['season', 'week']
].copy()

rb_chronological_order = rb_train_info.sort_values(
    ['season', 'week']
).index

X_train_rb_chrono = X_train_rb.loc[rb_chronological_order]
y_train_rb_chrono = y_train_rb.loc[rb_chronological_order]

In [131]:
# ----------- RB Expanding Season Validation Folds -----------

rb_season_series = rb_df.loc[
    X_train_rb_chrono.index,
    'season'
]

rb_cv_splits = []

for val_season in [2021, 2022, 2023]:
    train_indices = rb_season_series[rb_season_series < val_season].index
    val_indices = rb_season_series[rb_season_series == val_season].index

    train_positions = X_train_rb_chrono.index.get_indexer(train_indices)
    val_positions = X_train_rb_chrono.index.get_indexer(val_indices)

    rb_cv_splits.append((train_positions, val_positions))

for i, (train_idx, val_idx) in enumerate(rb_cv_splits, start=1):
    print(
        f"Fold {i}:",
        len(train_idx),
        "train rows,",
        len(val_idx),
        "validation rows"
    )

Fold 1: 2739 train rows, 1429 validation rows
Fold 2: 4168 train rows, 1494 validation rows
Fold 3: 5662 train rows, 1391 validation rows


In [134]:
# ----------- RB XGBoost Grid Search -----------

rb_param_grid = {
    'n_estimators': [200, 300],
    'learning_rate': [0.03, 0.05],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

rb_xgb_grid = GridSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_grid=rb_param_grid,
    scoring='neg_mean_absolute_error',
    cv=rb_cv_splits,
    n_jobs=-1
)

rb_xgb_grid.fit(
    X_train_rb_chrono,
    y_train_rb_chrono
)

print("Best RB parameters:", rb_xgb_grid.best_params_)
print("Best RB CV MAE:", -rb_xgb_grid.best_score_)

best_rb_xgb_model = rb_xgb_grid.best_estimator_

y_val_pred_best_rb_xgb = best_rb_xgb_model.predict(X_val_rb)

best_rb_xgb_val_mae = mean_absolute_error(
    y_val_rb,
    y_val_pred_best_rb_xgb
)

best_rb_xgb_val_rmse = root_mean_squared_error(
    y_val_rb,
    y_val_pred_best_rb_xgb
)

print("Best RB XGBoost 2024 Validation MAE:", best_rb_xgb_val_mae)
print("Best RB XGBoost 2024 Validation RMSE:", best_rb_xgb_val_rmse)

Best RB parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200, 'subsample': 0.8}
Best RB CV MAE: 4.842890575911155
Best RB XGBoost 2024 Validation MAE: 4.561187965171935
Best RB XGBoost 2024 Validation RMSE: 6.146295644444655


In [138]:
# ----------- QB Training, Validation, Test Split -----------

qb_seasons = qb_df.loc[X_qb.index, 'season']

train_mask_qb = qb_seasons <= 2023
val_mask_qb = qb_seasons == 2024
test_mask_qb = qb_seasons == 2025

X_train_qb = X_qb[train_mask_qb]
y_train_qb = y_qb[train_mask_qb]

X_val_qb = X_qb[val_mask_qb]
y_val_qb = y_qb[val_mask_qb]

X_test_qb = X_qb[test_mask_qb]
y_test_qb = y_qb[test_mask_qb]

print(X_train_qb.shape)
print(X_val_qb.shape)
print(X_test_qb.shape)

(2903, 25)
(618, 25)
(611, 25)


In [139]:
# ----------- QB Linear Regression Model -----------

qb_linear_model = LinearRegression()

qb_linear_model.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_qb = qb_linear_model.predict(X_val_qb)

mae_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_qb
)

rmse_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_qb
)

print("QB Linear Regression Validation MAE:", mae_qb)
print("QB Linear Regression Validation RMSE:", rmse_qb)

QB Linear Regression Validation MAE: 6.255603298028659
QB Linear Regression Validation RMSE: 7.882630931780585


In [140]:
# ----------- QB Random Forest Model -----------

qb_rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

qb_rf_model.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_rf_qb = qb_rf_model.predict(X_val_qb)

mae_rf_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_rf_qb
)

rmse_rf_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_rf_qb
)

print("QB Random Forest Validation MAE:", mae_rf_qb)
print("QB Random Forest Validation RMSE:", rmse_rf_qb)

QB Random Forest Validation MAE: 6.19332189798066
QB Random Forest Validation RMSE: 7.849615114320782


In [141]:
# ----------- QB XGBoost Model -----------

qb_xgb_model = XGBRegressor(
    n_estimators=100,
    random_state=42
)

qb_xgb_model.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_xgb_qb = qb_xgb_model.predict(X_val_qb)

mae_xgb_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_xgb_qb
)

rmse_xgb_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_xgb_qb
)

print("QB XGBoost Validation MAE:", mae_xgb_qb)
print("QB XGBoost Validation RMSE:", rmse_xgb_qb)

QB XGBoost Validation MAE: 6.747634002716797
QB XGBoost Validation RMSE: 8.566733340624337


In [142]:
# ----------- QB Tuned Random Forest -----------

qb_rf_tuned_1 = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)

qb_rf_tuned_1.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_rf_tuned_qb = qb_rf_tuned_1.predict(X_val_qb)

mae_rf_tuned_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_rf_tuned_qb
)

rmse_rf_tuned_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_rf_tuned_qb
)

print("Tuned QB Random Forest MAE:", mae_rf_tuned_qb)
print("Tuned QB Random Forest RMSE:", rmse_rf_tuned_qb)

Tuned QB Random Forest MAE: 6.150337641115546
Tuned QB Random Forest RMSE: 7.813471678295539


In [ ]:
# ----------- QB Tuned XGBoost -----------

qb_xgb_tuned_1 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

qb_xgb_tuned_1.fit(
    X_train_qb,
    y_train_qb
)

y_val_pred_xgb_tuned_qb = qb_xgb_tuned_1.predict(X_val_qb)

mae_xgb_tuned_qb = mean_absolute_error(
    y_val_qb,
    y_val_pred_xgb_tuned_qb
)

rmse_xgb_tuned_qb = root_mean_squared_error(
    y_val_qb,
    y_val_pred_xgb_tuned_qb
)

print("Tuned QB XGBoost MAE:", mae_xgb_tuned_qb)
print("Tuned QB XGBoost RMSE:", rmse_xgb_tuned_qb)